# Requirements Ambiguity Detector — v2
**Architecture:** RAG-style few-shot prompting with semantic embeddings + chain-of-thought reasoning

Improvements over v1:
- General ambiguity taxonomy (not just pronouns)
- Chain-of-thought reasoning in prompt
- Real explanation labels derived from dataset
- Full metrics: accuracy, F1, confusion matrix
- Robust JSON parsing with fallback keyword detection
- Plug-and-play for new general ambiguity dataset

## 0. Install dependencies

In [4]:
# Run once
!pip install sentence-transformers scikit-learn pandas openpyxl nltk


   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   -------------------------------


[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Imports

In [5]:
import pandas as pd
import requests
import json
import re
import time
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
from sentence_transformers import SentenceTransformer, util
import warnings
warnings.filterwarnings('ignore')
print("Imports OK")

c:\temp\amb_checker_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [6]:
# Match the preprocessing used by the backend ML classifier.
try:
    ML_STOPWORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords", quiet=True)
    ML_STOPWORDS = set(stopwords.words("english"))

ML_STOPWORDS.update({
    "set", "given", "shall", "be", "such", "as", "a", "an", "in",
    "the", "na", "would",
})
ML_STEMMER = PorterStemmer()

def preprocess_requirement(text: str) -> str:
    """Apply the backend ML pipeline's text cleaning to one requirement."""
    if pd.isna(text):
        return ""
    words = re.sub(r"[^a-zA-Z]", " ", str(text)).lower().split()
    return " ".join(
        ML_STEMMER.stem(word) for word in words if word not in ML_STOPWORDS
    )

print("ML-style requirement preprocessing ready.")

ML-style requirement preprocessing ready.


## 2. Configuration — change only this cell

In [7]:
# ─── FILE ────────────────────────────────────────────────────────────────────
DATA_FILE       = "requirements_dataset.xlsx"      # swap to your general ambiguity file when ready
TEXT_COL        = "Context"       # column holding the requirement text
MODEL_TEXT_COL  = "_model_text"   # cleaned text used for retrieval and LLM prompts
LABEL_COL       = "DetectedAs"    # column holding the label
AMBIG_LABEL     = "ambiguous"     # exact string used for ambiguous class (lowercased)

# ─── SPLIT ───────────────────────────────────────────────────────────────────
PROMPT_DEV_FRAC = 0.15            # fraction used to build the retrieval index
RANDOM_STATE    = 42

# ─── RETRIEVAL ───────────────────────────────────────────────────────────────
EMBED_MODEL     = 'all-MiniLM-L6-v2'   # fast & good; swap for 'all-mpnet-base-v2' for quality
K_EXAMPLES      = 5               # how many few-shot examples to retrieve

# ─── LLM ─────────────────────────────────────────────────────────────────────
OLLAMA_URL      = "http://localhost:11434/api/generate"
OLLAMA_MODEL    = "mistral"       # change to llama3, gemma2, etc.

# ─── EVAL ────────────────────────────────────────────────────────────────────
# Set to None to evaluate the full eval set (slow); set to e.g. 100 for a quick run
EVAL_SAMPLE_N   = 100

## 3. Load & inspect data

In [10]:
df = pd.read_excel(DATA_FILE)
df = df.dropna(subset=[TEXT_COL, LABEL_COL])
df[TEXT_COL]  = df[TEXT_COL].astype(str).str.strip()
df[MODEL_TEXT_COL] = df[TEXT_COL].map(preprocess_requirement)
df[LABEL_COL] = df[LABEL_COL].str.lower().str.strip()

print(f"Total samples : {len(df)}")
print(f"Label distribution:\n{df[LABEL_COL].value_counts()}")
print(f"\nSample row:")
display(df.head(3))

Total samples : 737
Label distribution:
DetectedAs
unambiguous    395
ambiguous      342
Name: count, dtype: int64

Sample row:


,Id,Context,Pronoun,DetectedAs,_model_text
0,HalifaxCombatSystems#1,"Under the Halifax-class WPC-W and WPC-E, DND w...",its,ambiguous,halifax class wpc w wpc e dnd chair canada ind...
1,HalifaxCombatSystems#2,Canada will use other organizations to support...,their,unambiguous,canada use organ support qualiti assur manag r...
2,HalifaxCombatSystems#3,The ship readiness levels and expected annual ...,its,unambiguous,ship readi level expect annual usag halifax cl...


In [11]:
df['word_count'] = df[TEXT_COL].apply(lambda x: len(x.split()))

print(f"Average word count : {df['word_count'].mean():.1f}")
print(f"Min word count     : {df['word_count'].min()}")
print(f"Max word count     : {df['word_count'].max()}")
print(f"Median word count  : {df['word_count'].median():.1f}")

# Distribution
print(f"\nWord count distribution:")
print(f"  < 20 words  : {(df['word_count'] < 20).sum()} requirements")
print(f"  20-50 words : {((df['word_count'] >= 20) & (df['word_count'] <= 50)).sum()} requirements")
print(f"  > 50 words  : {(df['word_count'] > 50).sum()} requirements")

Average word count : 47.4
Min word count     : 15
Max word count     : 119
Median word count  : 44.0

Word count distribution:
  < 20 words  : 9 requirements
  20-50 words : 451 requirements
  > 50 words  : 277 requirements


## 4. Train / test split

In [12]:
prompt_dev_df, eval_df = train_test_split(
    df,
    test_size   = 1 - PROMPT_DEV_FRAC,
    stratify    = df[LABEL_COL],
    random_state= RANDOM_STATE
)
prompt_dev_df = prompt_dev_df.reset_index(drop=True)
eval_df       = eval_df.reset_index(drop=True)

print(f"Prompt dev : {len(prompt_dev_df)} samples")
print(f"Eval set   : {len(eval_df)} samples")

Prompt dev : 110 samples
Eval set   : 627 samples


In [13]:
print("Prompt dev label distribution:")
print(prompt_dev_df[LABEL_COL].value_counts())

print("\nEval set label distribution:")
print(eval_df[LABEL_COL].value_counts())

Prompt dev label distribution:
DetectedAs
unambiguous    59
ambiguous      51
Name: count, dtype: int64

Eval set label distribution:
DetectedAs
unambiguous    336
ambiguous      291
Name: count, dtype: int64


## 5. Build semantic retrieval index

In [14]:
print(f"Loading embedding model: {EMBED_MODEL} ...")
embed_model = SentenceTransformer(EMBED_MODEL)

dev_texts  = prompt_dev_df[MODEL_TEXT_COL].tolist()
dev_labels = prompt_dev_df[LABEL_COL].tolist()

print("Encoding prompt-dev set ...")
X_dev = embed_model.encode(dev_texts, convert_to_tensor=True, show_progress_bar=True)
print("Index ready.")


def retrieve_examples(query: str, k: int = K_EXAMPLES) -> pd.DataFrame:
    q_vec   = embed_model.encode(query, convert_to_tensor=True)
    scores  = util.cos_sim(q_vec, X_dev)[0]
    top_idx = scores.argsort(descending=True)[:k * 4]  # wider candidate pool
    candidates = prompt_dev_df.iloc[top_idx.cpu().numpy()]

    # Always show equal ambiguous / unambiguous examples
    half    = k // 2
    ambig   = candidates[candidates[LABEL_COL] == 'ambiguous'].head(half)
    unambig = candidates[candidates[LABEL_COL] == 'unambiguous'].head(k - half)
    balanced = pd.concat([ambig, unambig])

    # Shuffle so the model doesn't see a pattern in ordering
    return balanced.sample(frac=1, random_state=42).reset_index(drop=True)

Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6042.40it/s]


Encoding prompt-dev set ...


Batches: 100%|██████████| 4/4 [00:00<00:00,  5.53it/s]

Index ready.


## 6. Ambiguity taxonomy

This taxonomy lets the prompt teach the model about *general* ambiguity, not just pronouns.
When your general ambiguity dataset arrives, add its categories here and the system prompt
will automatically cover them.

In [15]:
AMBIGUITY_TAXONOMY = """\
Types of ambiguity in software requirements:

1. PRONOUN AMBIGUITY
   Pronouns (it, they, their, its, this, these) whose referents are unclear.
   Example: "The system shall notify the user when it detects an error." → What does 'it' refer to?

2. VAGUE / WEAK LANGUAGE
   Words like 'fast', 'easy', 'appropriate', 'sufficient', 'adequate', 'user-friendly',
   'simple', 'flexible', 'robust' that have no measurable definition.
   Example: "The system shall respond quickly." → How quickly?

3. MISSING SUBJECT / ACTOR
   It is unclear WHO or WHAT performs an action.
   Example: "Errors shall be logged." → Logged by whom? When?

4. COORDINATION AMBIGUITY
   'and' / 'or' that can be grouped in multiple ways.
   Example: "The system shall support PDF and Word or Excel exports." → (PDF and Word) or Excel? or PDF and (Word or Excel)?

5. SCOPE / QUANTIFIER AMBIGUITY
   Unclear scope: 'all', 'each', 'any', 'every', 'some', 'most'.
   Example: "Each user shall have access to all reports." → Does 'all' include restricted reports?

6. TEMPORAL AMBIGUITY
   Unclear timing: 'when', 'after', 'before', 'during', 'immediately' with no defined threshold.
   Example: "The system shall update the record after approval." → How long after?

7. CONDITIONAL AMBIGUITY
   Conditions that can be interpreted in multiple ways.
   Example: "If the user is logged in and idle or disconnected, the session shall expire."

8. NEGATION AMBIGUITY
   Negations that create unclear scope.
   Example: "The system shall not allow users without admin rights and supervisors to delete records."
"""

## 7. Few-shot example builder

**Key fix vs v1:** explanations are NOT hardcoded templates.
We infer the ambiguity type from the actual text using simple heuristics,
producing richer, diverse few-shot examples.

In [16]:
# Keyword-based heuristics for ambiguity type detection
_PRONOUN_RE   = re.compile(r'\b(it|they|their|its|this|these|those|them)\b', re.I)
_VAGUE_RE     = re.compile(r'\b(fast|quick|quickly|easy|easily|appropriate|sufficient|adequate|user.friendly|simple|flexible|robust|efficient|reasonable|timely|high.quality|good|bad|large|small|many|few)\b', re.I)
_COORD_RE     = re.compile(r'\b(and|or)\b.*\b(and|or)\b', re.I)
_TEMPORAL_RE  = re.compile(r'\b(immediately|soon|quickly|eventually|when|before|after|during|within|upon)\b', re.I)
_SCOPE_RE     = re.compile(r'\b(all|every|each|any|some|most|several|multiple)\b', re.I)
_NEGATION_RE  = re.compile(r'\b(not|without|except|unless|no)\b', re.I)


def infer_ambiguity_type(text: str) -> str:
    """Return a descriptive explanation of why a requirement is ambiguous."""
    reasons = []
    if _PRONOUN_RE.search(text):
        pronoun = _PRONOUN_RE.search(text).group(0)
        reasons.append(f"pronoun '{pronoun}' has an unclear referent")
    if _VAGUE_RE.search(text):
        word = _VAGUE_RE.search(text).group(0)
        reasons.append(f"vague/unmeasurable term '{word}' is used")
    if _COORD_RE.search(text):
        reasons.append("coordination with 'and/or' allows multiple groupings")
    if _TEMPORAL_RE.search(text) and not _VAGUE_RE.search(text):
        reasons.append("temporal condition has no defined threshold")
    if _SCOPE_RE.search(text):
        word = _SCOPE_RE.search(text).group(0)
        reasons.append(f"quantifier '{word}' is undefined in scope")
    if _NEGATION_RE.search(text) and not reasons:
        reasons.append("negation creates unclear scope")

    if reasons:
        return "Ambiguous because " + "; ".join(reasons) + "."
    else:
        return "The requirement is ambiguous due to vague or imprecise language that admits multiple interpretations."


def build_examples(examples_df: pd.DataFrame) -> str:
    """Build a few-shot block from retrieved examples."""
    parts = []
    for _, row in examples_df.iterrows():
        text  = row[MODEL_TEXT_COL]
        label = row[LABEL_COL]

        if label == AMBIG_LABEL:
            explanation = infer_ambiguity_type(row[TEXT_COL])
        else:
            explanation = (
                "The requirement is specific and measurable; "
                "a developer can implement exactly one behaviour."
            )

        parts.append(
            f"Requirement: {text}\n"
            f"Classification: {label}\n"
            f"Explanation: {explanation}"
        )

    return "\n\n".join(parts)


print("Example builder ready.")

Example builder ready.


## 8. Prompt builder — chain-of-thought

In [17]:
def build_prompt(query: str, examples_text: str) -> str:
    return f"""\
You are an expert software requirements engineer specialising in ambiguity analysis.

══ AMBIGUITY TAXONOMY ══
{AMBIGUITY_TAXONOMY}

══ TASK ══
Analyse the given requirement. Think step-by-step:
  Step 1 — Identify every word, phrase, pronoun, or structure that could be unclear.
  Step 2 — Decide which ambiguity type(s) from the taxonomy apply (if any).
  Step 3 — Classify as "ambiguous" or "unambiguous".
  Step 4 — Write one concise explanation sentence.

══ WHAT IS NOT AMBIGUOUS ══
- Passive voice ("shall be conditioned", "shall be qualified", "shall be provided",
  "shall be avoided") is STANDARD in requirements engineering — do NOT flag as missing_actor.
- Technical acronyms and domain terms (MPAS, ERMS, CITIS, GSTSIM, EME, DMS, FTP, SPI)
  are NOT ambiguous on their own.
- A requirement with multiple clauses is NOT automatically ambiguous.
- Words like "other", "related", "associated", "respective" are NOT ambiguous
  unless their referent is genuinely unclear.

══ WHAT IS AMBIGUOUS ══
- Pronouns (it, they, their, its, this, these) where the referent is genuinely unclear.
- Unmeasurable terms (fast, soon, adequate, sufficient, appropriate, user-friendly)
  with no defined metric or threshold.
- Coordination (and/or) that allows multiple groupings with different meanings.
- Missing actor ONLY when it is impossible to infer who performs the action
  even from context — passive voice alone is NOT enough.
- Temporal conditions with no defined time threshold.
  
══ LABELLED EXAMPLES (similar to the test requirement) ══
{examples_text}

══ RULES ══
- Output ONLY valid JSON — no markdown, no extra text before or after.
- "classification" must be exactly "ambiguous" or "unambiguous".
- "ambiguity_type" must be one of: pronoun, vague_language, missing_actor, coordination,
  scope_quantifier, temporal, conditional, negation, none.
- "explanation" must be one clear sentence stating the specific issue found
  or confirming why the requirement is clear.
- BEFORE outputting, ask yourself: "Can I point to one specific word or phrase
  that would cause two developers to implement this differently?" 
  If yes → ambiguous. If no → unambiguous.

══ OUTPUT FORMAT ══
{{"classification": "ambiguous or unambiguous", "ambiguity_type": "type or none", "explanation": "one sentence"}}

══ REQUIREMENT TO CLASSIFY ══
"{query}"
"""


## 9. LLM call + robust JSON parser

In [18]:
def call_ollama(prompt: str) -> str:
    """Call local Ollama API. Returns raw string response."""
    response = requests.post(
        OLLAMA_URL,
        json={"model": OLLAMA_MODEL, "prompt": prompt, "stream": False},
        timeout=120
    )
    response.raise_for_status()
    return response.json()["response"]


def extract_json(raw: str) -> dict | None:
    """Try multiple strategies to parse JSON from LLM output."""
    # Strategy 1: direct parse
    try:
        return json.loads(raw.strip())
    except Exception:
        pass

    # Strategy 2: extract first {...} block
    match = re.search(r'\{[^{}]+\}', raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except Exception:
            pass

    # Strategy 3: look for JSON inside markdown code fences
    fence = re.search(r'```(?:json)?\s*([\s\S]*?)```', raw)
    if fence:
        try:
            return json.loads(fence.group(1).strip())
        except Exception:
            pass

    return None


def keyword_fallback(raw: str) -> str | None:
    """
    Last-resort: if JSON parsing fails entirely, scan the raw text
    for the classification keyword so we don't lose the sample.
    """
    raw_lower = raw.lower()
    if '"unambiguous"' in raw_lower or 'unambiguous' in raw_lower:
        return 'unambiguous'
    if '"ambiguous"' in raw_lower or 'ambiguous' in raw_lower:
        return 'ambiguous'
    return None


print("LLM helpers ready.")

LLM helpers ready.


## 10. Main evaluation loop

In [19]:
sample_df = eval_df if EVAL_SAMPLE_N is None else eval_df.iloc[:EVAL_SAMPLE_N]

actuals, preds, explanations, ambi_types = [], [], [], []
failed_indices = []
total_time = 0.0

for i, row in sample_df.iterrows():
    requirement = row[TEXT_COL]
    model_requirement = row[MODEL_TEXT_COL]
    actual      = row[LABEL_COL]

    retrieved     = retrieve_examples(model_requirement, k=K_EXAMPLES)
    examples_text = build_examples(retrieved)
    prompt        = build_prompt(model_requirement, examples_text)

    t0  = time.perf_counter()
    try:
        raw = call_ollama(prompt)
    except Exception as e:
        print(f"[{i}] API error: {e}")
        failed_indices.append(i)
        continue
    elapsed = time.perf_counter() - t0
    total_time += elapsed

    parsed = extract_json(raw)

    if parsed:
        predicted    = parsed.get("classification", "").lower().strip()
        explanation  = parsed.get("explanation", "")
        ambi_type    = parsed.get("ambiguity_type", "unknown")
    else:
        # Keyword fallback
        predicted = keyword_fallback(raw)
        if predicted is None:
            print(f"[{i}] Could not parse → skipping. Raw: {raw[:120]}")
            failed_indices.append(i)
            continue
        explanation = raw[:150].replace('\n', ' ')
        ambi_type   = "unknown"

    actuals.append(actual)
    preds.append(predicted)
    explanations.append(explanation)
    ambi_types.append(ambi_type)

    correct_mark = "✓" if predicted == actual else "✗"
    print(f"[{correct_mark}] #{i} | actual={actual:<12} predicted={predicted:<12} | {requirement[:80]}")

print(f"\nDone. Failed to parse: {len(failed_indices)} samples")

[✗] #0 | actual=ambiguous    predicted=unambiguous  | The MPAS shall display GUIs, textual dialogues, and content in Canadian English.
[✓] #1 | actual=ambiguous    predicted=ambiguous    | The ERMS must allow an authorised user to add new classes to the classification 
[✓] #2 | actual=ambiguous    predicted=ambiguous    | This may be even higher if the useable data rate for the forward tasking data is
[✓] #3 | actual=unambiguous  predicted=unambiguous  | The ERMS must be able to export physical folders, and retain all their associati
[✓] #4 | actual=ambiguous    predicted=ambiguous    | It shall be possible for the equipment to display the identity of the called or 
[✓] #5 | actual=unambiguous  predicted=unambiguous  | Maintenance activities assigned to the FMFs are defined in the HCCS EG maintenan
[✗] #6 | actual=unambiguous  predicted=ambiguous    | The MPAS shall transform data, when requested by a user, from other geodetic sys
[✗] #7 | actual=unambiguous  predicted=ambiguous    | T

## 11. Metrics

In [20]:
if not actuals:
    print("No predictions to evaluate.")
else:
    acc  = accuracy_score(actuals, preds)
    f1   = f1_score(actuals, preds, pos_label=AMBIG_LABEL, average='binary')
    prec = precision_score(actuals, preds, pos_label=AMBIG_LABEL, average='binary')
    rec  = recall_score(actuals, preds, pos_label=AMBIG_LABEL, average='binary')
    avg_t = total_time / len(actuals)

    print("═" * 50)
    print(f"Samples evaluated : {len(actuals)}  (failed: {len(failed_indices)})")
    print(f"Accuracy          : {acc:.4f}  ({acc*100:.1f}%)")
    print(f"F1 (ambiguous)    : {f1:.4f}")
    print(f"Precision         : {prec:.4f}")
    print(f"Recall            : {rec:.4f}")
    print(f"Avg time / sample : {avg_t:.2f}s")
    print("═" * 50)

    print("\nFull classification report:")
    print(classification_report(actuals, preds))

    print("Confusion matrix (rows=actual, cols=predicted):")
    labels_order = sorted(set(actuals + preds))
    cm = confusion_matrix(actuals, preds, labels=labels_order)
    cm_df = pd.DataFrame(cm, index=[f'actual_{l}' for l in labels_order],
                             columns=[f'pred_{l}' for l in labels_order])
    display(cm_df)

    print("\nPredicted ambiguity type distribution:")
    print(Counter(ambi_types))

══════════════════════════════════════════════════
Samples evaluated : 100  (failed: 0)
Accuracy          : 0.5100  (51.0%)
F1 (ambiguous)    : 0.5149
Precision         : 0.5000
Recall            : 0.5306
Avg time / sample : 57.40s
══════════════════════════════════════════════════

Full classification report:
              precision    recall  f1-score   support

   ambiguous       0.50      0.53      0.51        49
 unambiguous       0.52      0.49      0.51        51

    accuracy                           0.51       100
   macro avg       0.51      0.51      0.51       100
weighted avg       0.51      0.51      0.51       100

Confusion matrix (rows=actual, cols=predicted):


,pred_ambiguous,pred_unambiguous
actual_ambiguous,26,23
actual_unambiguous,26,25



Predicted ambiguity type distribution:
Counter({'none': 57, 'missing_actor': 12, 'coordination': 11, 'vague_language': 9, 'temporal': 8, 'scope_quantifier': 1, 'pronoun': 1, 'coordination, temporal, pronoun': 1})


## 12. Save results

In [76]:
results_df = sample_df.iloc[:len(actuals)].copy()
results_df['predicted']     = preds
results_df['explanation']   = explanations
results_df['ambiguity_type']= ambi_types
results_df['correct']       = [a == p for a, p in zip(actuals, preds)]

out_path = "results_v2.xlsx"
results_df.to_excel(out_path, index=False)
print(f"Results saved to {out_path}")
display(results_df.head(10))

Results saved to results_v2.xlsx


,Id,Context,Pronoun,DetectedAs,word_count,predicted,explanation,ambiguity_type,correct
0,MHP#624,"The MPAS shall display GUIs, textual dialogues...",its,ambiguous,35,ambiguous,The requirement is ambiguous because 'after' h...,temporal,True
1,ElectronicRecordsMgmtSystems#70,The ERMS must allow an authorised user to add ...,its,ambiguous,81,unambiguous,The requirement is specific and measurable; a ...,none,True
2,EDRS_MOCReqs#701,This may be even higher if the useable data ra...,its,ambiguous,35,ambiguous,The coordination 'or' allows multiple grouping...,coordination,True
3,ElectronicRecordsMgmtSystems#151,The ERMS must be able to export physical folde...,their,unambiguous,47,unambiguous,The requirement is specific and measurable; a ...,none,True
4,2007-eirene_fun_7-2#42b,It shall be possible for the equipment to disp...,their,ambiguous,46,unambiguous,The requirement is specific and measurable; a ...,none,True
5,HalifaxCombatSystems#54,Maintenance activities assigned to the FMFs ar...,their,unambiguous,40,unambiguous,The requirement is specific and measurable; a ...,none,True
6,MHP#629,"The MPAS shall transform data, when requested ...",its,unambiguous,45,unambiguous,The requirement is specific and measurable; a ...,none,True
7,EARS_NuclearReqs#443,The licence applicant/holder shall have a set ...,its,unambiguous,19,unambiguous,The requirement is specific and measurable; a ...,none,True
8,EARS_NuclearReqs#448,The life-cycle model of 315.1 shall consist of...,its,unambiguous,36,unambiguous,The requirement is specific and measurable; a ...,none,True
9,2007-eirene_fun_7-2#99,Prompts and messages shall be capable of being...,his,unambiguous,44,unambiguous,The requirement is specific and measurable; a ...,none,True


In [77]:
wrong_df  = results_df[~results_df['correct']]
fn        = wrong_df[wrong_df['predicted'] == 'unambiguous']
print(f"False Negatives: {len(fn)}")
for _, r in fn.iterrows():
    print(f"REQ: {r[TEXT_COL]}")
    print()

False Negatives: 21
REQ: The subsystem shall condition, control, store and distribute electrical power on the ROSETTA spacecraft. The subsystem shall provide adequate status monitoring and telecommand interfaces necessary to operate the subsystem and permit evaluation of its performance.

REQ: GSTSIM SIMAMC shall allow the generation of an availability schedule and its distribution to UHD via GSTSIM SIMAMC shall allow the generation of KPI Reports and its distribution to UHD via

REQ: The Contractor must maintain and calibrate the STTE used to support the HCCS EG systems. DND is responsible to hold STTE required for their assigned tasks and to report any maintenance and calibration requirements to the ISS Contractor.

REQ: fulfil an active rather than merely descriptive purpose in achieving that functionality automatically). The ERMS must support inheritance of metadata by folders allocated to a class so that, by default, addition of a new folder results in automatic inclusion of those

In [78]:
# Look at some ambiguous requirements manually
ambig_df = df[df[LABEL_COL] == 'ambiguous'].sample(20, random_state=42)
for _, r in ambig_df.iterrows():
    print(f"REQ: {r[TEXT_COL]}")
    print()

REQ: The IETIS shall display the approved nomenclature for a component at the first occurrence in a displayed text block. The IETIS shall consistently use the approved nomenclature or its associated standard shortened identifier.

REQ: All CSP equipment shall support the following climatic conditions in the off state in transport : -20c to +60c; normal atmospheric pressure; 10% to 90% relative humidity; transport in altitude : 0 to 8000 m. All CSP equipment delivered shall meet the acoustic noise pressure level applicable in the countries in which they will be installed.

REQ: The Rosetta software maintenance environment shall be integrated with the SVF. New software, or software patches to be uplinked to the spacecraft will need to be formally tested and verified prior to their release and use.

REQ: Where records are stored on re-writeable media, the ERMS must enable the complete obliteration34 of records, parts, folders, and groups of folders that have been so scheduled and confirme

## 13. Failure analysis — where does the model go wrong?

In [79]:
wrong_df = results_df[~results_df['correct']]
print(f"Wrong predictions: {len(wrong_df)}")

print("\n─── False Positives (predicted ambiguous, actually unambiguous) ───")
fp = wrong_df[(wrong_df['predicted'] == AMBIG_LABEL)]
for _, r in fp.head(5).iterrows():
    print(f"  REQ: {r[TEXT_COL]}")
    print(f"  EXP: {r['explanation']}\n")

print("\n─── False Negatives (predicted unambiguous, actually ambiguous) ───")
fn = wrong_df[(wrong_df['predicted'] != AMBIG_LABEL)]
for _, r in fn.head(5).iterrows():
    print(f"  REQ: {r[TEXT_COL]}")
    print(f"  EXP: {r['explanation']}\n")

Wrong predictions: 34

─── False Positives (predicted ambiguous, actually unambiguous) ───
  REQ: For quantitative calculations, 12000 hours shall be used as total vehicle lifetime operating hours. The MCCM shall have no potential incidents within its target life when subjected to any combination of usage and environment stated in this specification.
  EXP: Ambiguous because it is unclear who performs the action of 'processing the uplink signal' and 'distributing them to the users for execution'.

  REQ: The Lunar Exploration Light Rover shall meet all requirements in relative humidity of 20% to 100%, with a goal of 3%. The Lunar Exploration Light Rover shall meet its performance requirements under partial or complete cloud cover.
  EXP: The requirement is ambiguous because it's unclear who is responsible for providing access clearance and office space and infrastructure for government personnel

  REQ: The S/C primary structure shall be qualified by test; other secondary structure or 

## 14. Interactive tester — classify any single requirement

In [80]:
def classify_requirement(requirement: str, verbose: bool = True) -> dict:
    """Classify a single new requirement and return the result dict."""
    model_requirement = preprocess_requirement(requirement)
    retrieved     = retrieve_examples(model_requirement)
    examples_text = build_examples(retrieved)
    prompt        = build_prompt(model_requirement, examples_text)

    raw    = call_ollama(prompt)
    parsed = extract_json(raw)

    if not parsed:
        fallback = keyword_fallback(raw)
        parsed = {"classification": fallback or "unknown",
                  "ambiguity_type": "unknown",
                  "explanation": raw[:200]}

    if verbose:
        print(f"Requirement   : {requirement}")
        print(f"Classification: {parsed['classification'].upper()}")
        print(f"Ambiguity type: {parsed.get('ambiguity_type', 'N/A')}")
        print(f"Explanation   : {parsed['explanation']}")

    return parsed


# ─── Quick demo ───────────────────────────────────────────────────────────────
test_requirements = [
    "The system shall respond quickly to user requests.",
    "The system shall return search results within 2 seconds for queries with up to 1000 results.",
    "The system shall allow admin users to delete any record from the database.",
    "When it detects an error, it shall notify the user and log the event."
]

for req in test_requirements:
    print("─" * 70)
    classify_requirement(req)

──────────────────────────────────────────────────────────────────────
Requirement   : The system shall respond quickly to user requests.
Classification: AMBIGUOUS
Ambiguity type: vague_language
Explanation   : The term 'quickly' does not have a measurable definition, leading to potential ambiguity.
──────────────────────────────────────────────────────────────────────
Requirement   : The system shall return search results within 2 seconds for queries with up to 1000 results.
Classification: UNAMBIGUOUS
Ambiguity type: none
Explanation   : The requirement is specific and measurable; a developer can implement exactly one behaviour.
──────────────────────────────────────────────────────────────────────
Requirement   : The system shall allow admin users to delete any record from the database.
Classification: UNAMBIGUOUS
Ambiguity type: none
Explanation   : The requirement is specific and measurable; a developer can implement exactly one behaviour.
─────────────────────────────────────────

---

## Upgrade paths — what to do when the general ambiguity dataset arrives

1. **Swap the data file**: change `DATA_FILE` in the config cell to your new Excel/CSV file.
   Ensure `TEXT_COL` and `LABEL_COL` match the new column names.

2. **Extend the taxonomy** in Section 6 with any new ambiguity types found in the new data.
   The prompt will automatically pick them up.

3. **Upgrade the LLM** for higher accuracy:
   - Change `OLLAMA_MODEL` to `llama3:70b` or `command-r-plus` for better reasoning.
   - Alternatively swap `call_ollama` to a `call_openai` or `call_anthropic` function.

4. **Fine-tune** (highest accuracy path): once you have 500+ labelled general examples,
   fine-tune a small model (e.g. Llama-3-8B or Mistral-7B) using LoRA/QLoRA.
   Use the explanations as the target `output` field for instruction tuning.
   Tools: `unsloth`, `axolotl`, or `trl`.

5. **Add a confidence threshold**: run the LLM twice on borderline cases
   and take a majority vote to reduce false positives.